# Curriculum analysis (RQ1a) – evaluation of the coding

**Input:** the coding export(s) of the viewer (`04_coding_viewer/codings/coding_<coder>.json`) and the corpus
(`03_extraction_table/passages.json`).

**Unit of coding:** the *statement* (a passage can contain several).
**Unit of aggregation:** the *state* (`Land`). A state counts once per category, however many documents or
statements it has – parallel documents (e.g. G8/G9) therefore cannot inflate counts. GI and KMK are
*recommendations*, not state curricula, and are reported separately.

**Bindingness order** (used whenever several statements meet): verbindlich > Empfehlung > Beispiel > nicht spezifiziert.

Sections
1. Configuration
2. Load and flatten the coding
3. Data quality
4. Corpus check (school type and document versions)
5. Overview: tool types by state and level (data for the thesis table)
6. Named programming tools
7. Requirements by grade
8. Progression, reasons, attributes
9. Recommendations (GI/KMK) vs. states
10. Export (Excel, LaTeX, figures)

## 1 Configuration
All choices that affect the results are made here.

In [ ]:
from pathlib import Path
import json, re
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

BASE = Path.cwd()
if BASE.name != "05_synthesis":          # allow running from the project root as well
    BASE = next(p for p in [BASE / "Curricula" / "05_synthesis", BASE / "05_synthesis"] if p.exists())
CURRICULA = BASE.parent
CODING_FILE = CURRICULA / "04_coding_viewer" / "codings" / "coding_rater1.json"
CORPUS_FILE = CURRICULA / "03_extraction_table" / "passages.json"
OUT = BASE / "output"
(OUT / "tex").mkdir(parents=True, exist_ok=True)
(OUT / "fig").mkdir(parents=True, exist_ok=True)

# Recommendation documents (not state curricula) -- reported separately
RECOMMENDATION_GROUPS = {"GI", "KMK"}

# Additional corpus decisions made here (on top of the viewer decision and corpus_include.csv).
# Key: document key as shown in section 4, value: True (include) / False (exclude). Document the reason.
# Corpus decisions are recorded with reasons in 00_sources/corpus_include.csv (columns include, reason);
# rule: general-education Gymnasium and comprehensive schools, newest version only.
# Overrides here are only for ad-hoc sensitivity checks (key: document key, value: True/False).
CORPUS_OVERRIDES = {}

# Grades assumed when a statement gives a level (Sek I / Sek II) but no explicit grade.
# These are approximations and are flagged (grade_source = "level").
SEK_I_GRADES = list(range(5, 11))            # 5-10
SEK_II_GRADES = [11, 12, 13]
SEK_I_GRADES_BY_STATE = {"Berlin&Brandenburg": list(range(7, 11)),   # primary school up to grade 6
                         "Berlin": list(range(7, 11)), "Brandenburg": list(range(7, 11))}

# Unit of analysis per document, where it differs from the folder (state) of the document.
# Berlin and Brandenburg share the Sek I curriculum (unit "Berlin&Brandenburg"), but have separate
# Sek II curricula, which are counted as two units.
DOC_STATE = {
    "Lehrpläne/Berlin&Brandenburg/rahmenlehrplan-informatik-go-teil-c": "Berlin",
    "Lehrpläne/Berlin&Brandenburg/BB-Teil_C_RLP_GOST_2022_Informatik": "Brandenburg",
}
STATE_LABEL = {"Berlin&Brandenburg": "Berlin/Brandenburg"}   # display names in tables

def state_of(doc):
    return DOC_STATE.get(doc, DOCS[doc]["Land"])
GRADE_AXIS = list(range(5, 14))

BINDING_ORDER = ["verbindlich", "Empfehlung", "Beispiel", "nicht spezifiziert"]
BINDING_SYMBOL = {"verbindlich": "✓", "Empfehlung": "+", "Beispiel": "x", "nicht spezifiziert": "?"}
BINDING_TEX = {"verbindlich": r"\checkmark", "Empfehlung": "+", "Beispiel": "x", "nicht spezifiziert": "?"}

TOOL_TYPE_CODES = {"blockbasiert": "block-based", "visuell": "visual", "textbasiert": "text-based"}
ATTRIBUTE_CODES = {"geeignet": "suitable/didactically reduced", "einfach": "simple", "altersangemessen": "age-appropriate",
                   "aktuell": "current/modern", "mittel_zum_zweck": "means to an end", "open_source": "free/open source",
                   "inhalt_proxy": "content as proxy (OOP/paradigm)", "progression": "transition/progression"}

# Named tools -> (category, PL/PE). Extend if new names appear (section 6 lists unmapped names).
TOOL_INFO = {
    "Scratch": ("block-based", "PL+PE"), "Blockly": ("block-based", "PL+PE"), "Snap!": ("block-based", "PL+PE"),
    "Open Roberta": ("block-based", "PE"), "MakeCode": ("block-based", "PE"), "App Inventor": ("block-based", "PE"),
    "Java": ("text: general purpose", "PL"), "Python": ("text: general purpose", "PL"), "C++": ("text: general purpose", "PL"),
    "C#": ("text: general purpose", "PL"), "Delphi": ("text: general purpose", "PL+PE"), "Pascal": ("text: general purpose", "PL"),
    "PHP": ("text: scripting/web", "PL"), "JavaScript": ("text: scripting/web", "PL"), "Perl": ("text: scripting/web", "PL"),
    "Prolog": ("text: logic/functional", "PL"), "Scheme": ("text: logic/functional", "PL"), "Lisp": ("text: logic/functional", "PL"),
    "Haskell": ("text: logic/functional", "PL"),
    "Logo": ("educational/micro-world", "PL+PE"), "Kara": ("educational/micro-world", "PL+PE"),
    "Robot Karol": ("educational/micro-world", "PL+PE"), "Hamster": ("educational/micro-world", "PL+PE"),
    "BlueJ": ("educational IDE", "PE"), "Greenfoot": ("educational IDE", "PE"), "Thonny": ("educational IDE", "PE"),
    "Calliope": ("physical computing", "hardware"), "Arduino": ("physical computing", "hardware"),
    "micro:bit": ("physical computing", "hardware"), "Raspberry Pi": ("physical computing", "hardware"),
    "LEGO Mindstorms": ("physical computing", "hardware"), "LEGO Spike": ("physical computing", "hardware"),
    "LEGO WeDo": ("physical computing", "hardware"),
}

def strongest(values):
    """Strongest bindingness among values ('' and None count as 'nicht spezifiziert')."""
    vals = [v if v in BINDING_ORDER else "nicht spezifiziert" for v in values]
    return min(vals, key=BINDING_ORDER.index) if vals else None

print("coding:", CODING_FILE.name, "| output:", OUT)

## 2 Load and flatten the coding
One row per statement. Corpus membership: `CORPUS_OVERRIDES` > decision in the viewer > `corpus_include.csv`.

In [ ]:
coding = json.loads(CODING_FILE.read_text(encoding="utf-8-sig"))
corpus = json.loads(CORPUS_FILE.read_text(encoding="utf-8"))
DOCS, PASS = corpus["documents"], {p["id"]: p for p in corpus["passages"]}
for pid, a in coding.get("added", {}).items():          # passages added manually in the viewer
    PASS[pid] = {"id": pid, "doc": a["doc"], "page": a["page"], "text": a.get("text", ""), "added": True}
assert coding.get("schema") == "curricula-coding-v2", "please re-export from the current viewer (schema v2)"

def in_corpus(doc):
    if doc in CORPUS_OVERRIDES:
        return CORPUS_OVERRIDES[doc]
    decision = (coding["documents"].get(doc) or {}).get("in_corpus", "")
    if decision in ("ja", "nein"):
        return decision == "ja"
    return DOCS[doc].get("in_corpus") is not False

def parse_grades(text):
    """'Jgst. 5/6' -> [5, 6]; 'Jgst 7-10' -> [7..10]; '12/13' -> [12, 13]. Only grades 5-13."""
    if not text:
        return []
    t = str(text)
    grades = set()
    for a, b in re.findall(r"(\d{1,2})\s*[-–]\s*(\d{1,2})", t):
        grades.update(range(int(a), int(b) + 1))
    t = re.sub(r"(\d{1,2})\s*[-–]\s*(\d{1,2})", " ", t)
    grades.update(int(n) for n in re.findall(r"\d{1,2}", t))
    return sorted(g for g in grades if 5 <= g <= 13)

def level_grades(level, land):
    sek1 = SEK_I_GRADES_BY_STATE.get(land, SEK_I_GRADES)
    return {"Sek I": sek1, "Sek II": SEK_II_GRADES, "Sek I+II": sek1 + SEK_II_GRADES}.get(level, [])

def tool_list(s):
    return [t.strip() for t in re.split(r"[;,]", s or "") if t.strip()]

rows, passage_rows = [], []
for pid, r in coding["passages"].items():
    p = PASS.get(pid)
    if p is None or not r.get("relevant"):
        continue
    d = DOCS[p["doc"]]
    meta = {"passage": pid, "doc": p["doc"], "Land": state_of(p["doc"]), "group": d["Gruppe"],
            "is_recommendation": d["Land"] in RECOMMENDATION_GROUPS, "document": d["Dokument"],
            "school_folder": d["Schulform/Ordner"], "page": p["page"], "in_corpus": in_corpus(p["doc"])}
    ss = r.get("statements", []) if r["relevant"] == "ja" else []
    passage_rows.append({**meta, "relevant": r["relevant"], "n_statements": len(ss), "note": r.get("note", "")})
    for i, s in enumerate(ss, 1):
        explicit = parse_grades(s.get("grades"))
        grades = explicit or level_grades(s.get("level"), state_of(p["doc"]))
        rows.append({**meta, "statement": i, "n_statements": len(ss),
                     **{f"code_{k}": bool(v) for k, v in s.get("codes", {}).items()},
                     "codes": [k for k, v in s.get("codes", {}).items() if v],
                     "tools": tool_list(s.get("tools")), "level": s.get("level") or "",
                     "grades_verbatim": s.get("grades", ""), "grades": grades,
                     "grade_source": "explicit" if explicit else ("level" if grades else "none"),
                     "course": s.get("course") or "", "niveau": s.get("niveau") or "",
                     "binding": s.get("binding") if s.get("binding") in BINDING_ORDER else "nicht spezifiziert",
                     "binding_coded": bool(s.get("binding")), "marker": s.get("marker", ""),
                     "reason": s.get("reason", ""), "quote": s.get("quote") or p.get("text", ""),
                     "quote_from_pdf": bool(s.get("quote")), "quote_checked": s.get("quote_checked", "")})

ALL = pd.DataFrame(rows)
PASSAGES_CODED = pd.DataFrame(passage_rows)
S = ALL[ALL.in_corpus].reset_index(drop=True)                    # statements in the corpus
STATES = S[~S.is_recommendation]                                  # state curricula only
REC = S[S.is_recommendation]                                      # GI / KMK
print(f"statements: {len(ALL)} total, {len(S)} in corpus ({len(STATES)} state curricula, {len(REC)} GI/KMK)")
print(f"states with at least one statement: {STATES.Land.nunique()}")

## 3 Data quality
Things to fix in the viewer before the results are final (re-export, then re-run).

In [ ]:
q = pd.Series({
    "passages coded (any relevance)": len(PASSAGES_CODED),
    "  relevant = ja": (PASSAGES_CODED.relevant == "ja").sum(),
    "  relevant = nein": (PASSAGES_CODED.relevant == "nein").sum(),
    "  relevant = unklar": (PASSAGES_CODED.relevant == "unklar").sum(),
    "statements (in corpus)": len(S),
    "  without level": (S.level == "").sum(),
    "  without bindingness": (~S.binding_coded).sum(),
    "  without level of demand (niveau)": (S.niveau == "").sum(),
    "  without course type": (S.course == "").sum(),
    "  with explicit grades": (S.grade_source == "explicit").sum(),
    "  grades only from level (approximation)": (S.grade_source == "level").sum(),
    "  no grade information at all": (S.grade_source == "none").sum(),
    "  quote taken from the PDF": S.quote_from_pdf.sum(),
    "  quote checked (ja/korrigiert)": S.quote_checked.isin(["ja", "korrigiert"]).sum(),
    "passages with several statements": (PASSAGES_CODED.n_statements > 1).sum(),
})
display(q.to_frame("n"))
unclear = PASSAGES_CODED[PASSAGES_CODED.relevant == "unklar"]
if len(unclear):
    print("\nPassages still coded 'unklar' (decide before finalising):")
    display(unclear[["Land", "document", "page", "note"]])
missing = S[(S.level == "") | (~S.binding_coded)]
if len(missing):
    print("\nStatements without level or bindingness:")
    display(missing[["Land", "document", "page", "statement", "codes", "level", "binding_coded", "quote"]])

# explicit grades that do not fit the coded level (e.g. level inherited from a wrong document default)
def level_conflict(r):
    allowed = set(level_grades(r.level, r.Land))
    return r.grade_source == "explicit" and bool(allowed) and not set(r.grades) <= allowed
conflicts = S[S.apply(level_conflict, axis=1)]
if len(conflicts):
    print(f"\n{len(conflicts)} statements whose explicit grades contradict the coded level (check level / document default):")
    display(conflicts[["Land", "document", "page", "statement", "level", "grades_verbatim", "course", "quote"]])

## 4 Corpus check
Rule: *Gymnasium where a state has several school types; otherwise the unified curriculum (if available).*
The table lists all documents in the corpus. Flags are **heuristics** based on document and folder names:

* `other school type?` – name suggests a non-Gymnasium school type (Stadtteilschule, Realschule, Gemeinschaftsschule, Sekundarschule, Mittel-/Hauptschule, Fachschule/vocational)
* `several docs same level` – the state has more than one included document for the same level (e.g. G8/G9 or old/new version)

Decide per flagged document and record the decision in the viewer (*Im Korpus?* + reason) or in `CORPUS_OVERRIDES` above.
State-level results (sections 5–7) count each state once, so parallel versions do not inflate counts – but an
outdated version can still add requirements that no longer apply.

In [ ]:
OTHER_TYPE = re.compile(r"\bsts\b|stadtteil|realschul|(^|[/_\-\s])rs[_\-]|gemeinschaftsschul|gems|sekundarschul|\bsks\b|_sks_|"
                        r"mittelschul|hauptschul|\bfs[_\s]|fachschul|berufs|fachober|wirtschaftsschul", re.I)
doc_rows = []
for key, d in DOCS.items():
    if not in_corpus(key):
        continue
    sub = S[S.doc == key]
    doc_rows.append({"doc": key, "Land": state_of(key), "document": d["Dokument"], "folder": d["Schulform/Ordner"],
                     "statements": len(sub), "levels": ", ".join(sorted(set(sub.level) - {""})),
                     "other school type?": bool(OTHER_TYPE.search(key)),
                     "decision in viewer": (coding["documents"].get(key) or {}).get("in_corpus", "") or "CSV",
                     "reason": (coding["documents"].get(key) or {}).get("corpus_reason", "")})
CORPUS_DOCS = pd.DataFrame(doc_rows).sort_values(["Land", "doc"])
lvl = S[S.level != ""].groupby(["Land", "level"]).doc.nunique()
multi = {(l, v) for (l, v), n in lvl.items() if n > 1}
CORPUS_DOCS["several docs same level"] = CORPUS_DOCS.apply(
    lambda r: any((r.Land, v) in multi for v in r.levels.split(", ") if v), axis=1)
print(f"{len(CORPUS_DOCS)} documents in corpus, {CORPUS_DOCS.statements.eq(0).sum()} without any statement")
flagged = CORPUS_DOCS[CORPUS_DOCS["other school type?"] | CORPUS_DOCS["several docs same level"]]
print(f"\n{len(flagged)} flagged documents – please check:")
display(flagged[["Land", "document", "folder", "statements", "levels", "other school type?", "several docs same level", "reason"]])

In [ ]:
# Full list of corpus documents (for the appendix of the thesis)
display(CORPUS_DOCS[["Land", "document", "folder", "statements", "levels"]])

## 5 Overview: tool types by state and level
For each state and level (Sek I / Sek II): the strongest bindingness with which a tool type is required
(✓ verbindlich, + Empfehlung, x Beispiel, ? not specified). Statements coded *Sek I+II* count for both levels.
Course-type restrictions (elective, LK) are listed in the column *restricted to*.

In [ ]:
def expand_levels(df):
    out = []
    for _, r in df.iterrows():
        for lv in ({"Sek I+II": ["Sek I", "Sek II"]}.get(r.level) or [r.level or "unklar"]):
            out.append({**r.to_dict(), "lvl": lv})
    return pd.DataFrame(out)

RESTRICTED_COURSES = {"Wahlpflicht/Wahlfach": "elective", "Leistungskurs": "LK"}

def is_restricted(r):
    return r.course in RESTRICTED_COURSES or r.niveau == "erhöht"

def cell(sel):
    """Symbol of the strongest bindingness. Statements valid for all students take precedence; if a
    requirement exists only for elective courses / LK / advanced level, this is added in parentheses."""
    if not len(sel):
        return ""
    restricted = sel.apply(is_restricted, axis=1)
    general = sel[~restricted]
    if len(general):
        return BINDING_SYMBOL[strongest(general.binding)]
    labels = sorted({RESTRICTED_COURSES[c] for c in sel.course if c in RESTRICTED_COURSES}
                    | ({"adv."} if (sel.niveau == "erhöht").any() else set()))
    return f"{BINDING_SYMBOL[strongest(sel.binding)]} ({'/'.join(labels)})"

def overview(df, codes):
    x = expand_levels(df)
    rows = []
    for (land, lv), g in x.groupby(["Land", "lvl"]):
        rows.append({"Land": land, "level": lv, **{label: cell(g[g[f"code_{c}"]]) for c, label in codes.items()}})
    return pd.DataFrame(rows)

OVERVIEW_TYPES = overview(STATES, TOOL_TYPE_CODES)
display(OVERVIEW_TYPES.pivot(index="Land", columns="level", values=list(TOOL_TYPE_CODES.values())).fillna(""))

In [ ]:
# Counts: number of states requiring a tool type per level (any bindingness / binding only)
x = expand_levels(STATES)
count_rows = []
for c, label in TOOL_TYPE_CODES.items():
    for lv in ["Sek I", "Sek II"]:
        sel = x[(x[f"code_{c}"]) & (x.lvl == lv)]
        count_rows.append({"tool type": label, "level": lv, "states (any)": sel.Land.nunique(),
                           "states (verbindlich)": sel[sel.binding == "verbindlich"].Land.nunique(),
                           "states (Empfehlung)": sel[sel.binding == "Empfehlung"].Land.nunique(),
                           "states (Beispiel)": sel[sel.binding == "Beispiel"].Land.nunique()})
TYPE_COUNTS = pd.DataFrame(count_rows)
# states mentioning BOTH a visual/block-based and a text-based tool type (transition pattern)
vis_block = set(STATES[STATES.code_visuell | STATES.code_blockbasiert].Land)
text = set(STATES[STATES.code_textbasiert].Land)
print(f"states with visual/block-based AND text-based statements: {len(vis_block & text)} -> {sorted(vis_block & text)}")
print(f"states with progression statements: {sorted(set(STATES[STATES.code_progression].Land))}")
display(TYPE_COUNTS)

In [ ]:
# Attributes required of tools (suitable, simple, age-appropriate, ...) by state
OVERVIEW_ATTR = overview(STATES, ATTRIBUTE_CODES)
attr_states = {label: STATES[STATES[f"code_{c}"]].Land.nunique() for c, label in ATTRIBUTE_CODES.items()}
display(pd.Series(attr_states, name="number of states").to_frame())
display(OVERVIEW_ATTR.pivot(index="Land", columns="level", values=list(ATTRIBUTE_CODES.values())).fillna(""))

## 6 Named programming tools

In [ ]:
tool_rows = []
for _, r in S.iterrows():
    for t in r.tools:
        cat, plpe = TOOL_INFO.get(t, ("UNMAPPED", "?"))
        tool_rows.append({"tool": t, "category": cat, "PL/PE": plpe, "Land": r.Land, "is_recommendation": r.is_recommendation,
                          "document": r.document, "page": r.page, "level": r.level, "grades": r.grades,
                          "grade_source": r.grade_source, "course": r.course, "niveau": r.niveau, "binding": r.binding,
                          "quote": r.quote})
TOOLS = pd.DataFrame(tool_rows)
unmapped = sorted(TOOLS[TOOLS.category == "UNMAPPED"].tool.unique())
if unmapped:
    print("Add these names to TOOL_INFO in section 1:", unmapped)
T_STATES = TOOLS[~TOOLS.is_recommendation]

TOOL_SUMMARY = (T_STATES.groupby(["tool", "category", "PL/PE"])
    .agg(states=("Land", "nunique"), statements=("Land", "size"),
         strongest=("binding", strongest),
         levels=("level", lambda s: ", ".join(sorted(set(s) - {""}))),
         state_list=("Land", lambda s: ", ".join(sorted(set(s)))))
    .reset_index())
binding_states = T_STATES[T_STATES.binding == "verbindlich"].groupby("tool").Land.nunique()
TOOL_SUMMARY.insert(4, "states_binding", TOOL_SUMMARY.tool.map(binding_states).fillna(0).astype(int))
TOOL_SUMMARY = TOOL_SUMMARY.sort_values(["states", "statements"], ascending=False).reset_index(drop=True)
display(TOOL_SUMMARY)

In [ ]:
# Tool x state matrix (strongest bindingness symbol), state curricula + GI/KMK
TOOL_MATRIX = (TOOLS.groupby(["tool", "Land"]).binding.agg(strongest).map(BINDING_SYMBOL)
               .unstack(fill_value="").reindex(TOOL_SUMMARY.tool.tolist() + sorted(set(TOOLS.tool) - set(TOOL_SUMMARY.tool))))
display(TOOL_MATRIX)

# Categories: number of states naming at least one tool of the category, by level
cat = (expand_levels(S[~S.is_recommendation & S.tools.map(bool)].assign(category=lambda d: d.tools.map(
        lambda ts: sorted({TOOL_INFO.get(t, ("UNMAPPED",))[0] for t in ts})))).explode("category"))
TOOL_CATEGORIES = cat.groupby(["category", "lvl"]).Land.nunique().unstack(fill_value=0)
display(TOOL_CATEGORIES)

In [ ]:
# Figure: named tools by number of states, stacked by strongest bindingness per state
per_state = T_STATES.groupby(["tool", "Land"]).binding.agg(strongest).reset_index()
fig_data = per_state.groupby(["tool", "binding"]).size().unstack(fill_value=0).reindex(columns=BINDING_ORDER, fill_value=0)
fig_data = fig_data.loc[fig_data.sum(axis=1).sort_values().index]
colors = {"verbindlich": "#2f6f4e", "Empfehlung": "#6aa684", "Beispiel": "#b9d7c5", "nicht spezifiziert": "#dddddd"}
fig, ax = plt.subplots(figsize=(7, 0.32 * len(fig_data) + 1))
left = None
for b in BINDING_ORDER:
    ax.barh(fig_data.index, fig_data[b], left=left, color=colors[b], label=b, edgecolor="white")
    left = fig_data[b] if left is None else left + fig_data[b]
ax.set_xlabel("number of states naming the tool")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.legend(title="strongest bindingness", frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(OUT / "fig" / f"named_tools.{ext}", dpi=200)
plt.show()

## 7 Requirements by grade
Each statement is assigned to grades: explicit grade information where coded (`grade_source = explicit`), otherwise
the grades of its level (`grade_source = level`, see `SEK_I_GRADES`/`SEK_II_GRADES` – an approximation).
Cells give the **number of states** with at least one statement for that grade.

In [ ]:
G = S.explode("grades").dropna(subset=["grades"]).astype({"grades": int})
G_STATES = G[~G.is_recommendation]
print("statements by grade source:", S.grade_source.value_counts().to_dict())

ROW_DEF = {**{label: f"code_{c}" for c, label in TOOL_TYPE_CODES.items()},
           **{label: f"code_{c}" for c, label in ATTRIBUTE_CODES.items() if c not in ("inhalt_proxy",)}}
def grade_matrix(df):
    m = {}
    for label, col in ROW_DEF.items():
        m[label] = df[df[col]].groupby("grades").Land.nunique()
    for c in sorted({TOOL_INFO.get(t, ("UNMAPPED",))[0] for ts in df.tools for t in ts}):
        sel = df[df.tools.map(lambda ts: any(TOOL_INFO.get(t, ("UNMAPPED",))[0] == c for t in ts))]
        m[f"named tool: {c}"] = sel.groupby("grades").Land.nunique()
    return pd.DataFrame(m).T.reindex(columns=GRADE_AXIS).fillna(0).astype(int)

GRADES_ALL = grade_matrix(G_STATES)
GRADES_BINDING = grade_matrix(G_STATES[G_STATES.binding == "verbindlich"])
GRADES_EXPLICIT = grade_matrix(G_STATES[G_STATES.grade_source == "explicit"])
print("All statements:"); display(GRADES_ALL)
print("Only 'verbindlich':"); display(GRADES_BINDING)
print("Only explicit grade information (no level approximation):"); display(GRADES_EXPLICIT)

In [ ]:
def heatmap(m, title, fname):
    m = m[m.sum(axis=1) > 0]
    fig, ax = plt.subplots(figsize=(0.55 * m.shape[1] + 3.5, 0.38 * m.shape[0] + 1.2))
    im = ax.imshow(m.values, cmap="Greens", aspect="auto", vmin=0)
    ax.set_xticks(range(m.shape[1]), [str(g) for g in m.columns]); ax.set_xlabel("grade")
    ax.set_yticks(range(m.shape[0]), m.index)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            if v:
                ax.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v > m.values.max() * 0.6 else "black")
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, label="number of states", shrink=0.8)
    plt.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(OUT / "fig" / f"{fname}.{ext}", dpi=200)
    plt.show()

heatmap(GRADES_ALL, "Requirements by grade (all statements; grades partly derived from level)", "grades_all")
heatmap(GRADES_BINDING, "Requirements by grade (only 'verbindlich')", "grades_binding")

In [ ]:
# Per state: which tool types (B = block-based, V = visual, T = text-based) appear at which grade
# lower case = only Beispiel/Empfehlung/not specified, upper case = verbindlich
letters = {"code_blockbasiert": "B", "code_visuell": "V", "code_textbasiert": "T"}
cells = defaultdict(set)
for _, r in G_STATES.iterrows():
    for col, L in letters.items():
        if r[col]:
            cells[(r.Land, r.grades)].add(L if r.binding == "verbindlich" else L.lower())
STATE_GRADES = pd.DataFrame(
    {g: {land: "".join(sorted(cells.get((land, g), set()), key=lambda c: "BVTbvt".index(c)))
         for land in sorted(G_STATES.Land.unique())} for g in GRADE_AXIS})
display(STATE_GRADES)

# Named tools by grade (states)
TOOLS_BY_GRADE = (T_STATES.explode("grades").dropna(subset=["grades"]).astype({"grades": int})
                  .groupby(["tool", "grades"]).Land.nunique().unstack(fill_value=0).reindex(columns=GRADE_AXIS, fill_value=0))
display(TOOLS_BY_GRADE)

In [ ]:

# Explicit grade information only (no level approximation): tool types per state and grade.
# B/V/T = block-based / visual / text-based; upper case = verbindlich, lower case = weaker bindingness.
# GI recommendations are added as a reference row. This is the basis for statements about grades in the thesis.
E = G[G.grade_source == "explicit"]
cells_e = defaultdict(set)
for _, r in E.iterrows():
    for col, L in letters.items():
        if r[col]:
            cells_e[(r.Land, r.grades)].add(L if r.binding == "verbindlich" else L.lower())
rows_e = sorted({land for land, _ in cells_e}, key=lambda l: (l in RECOMMENDATION_GROUPS, l))
EXPLICIT_GRADES = pd.DataFrame(
    {g: {land: "".join(sorted(cells_e.get((land, g), set()), key=lambda c: "BVTbvt".index(c))) for land in rows_e}
     for g in GRADE_AXIS})
EXPLICIT_GRADES = EXPLICIT_GRADES.loc[:, (EXPLICIT_GRADES != "").any()]
display(EXPLICIT_GRADES)
print("states with explicit grade information on tool types:", sorted(set(rows_e) - RECOMMENDATION_GROUPS))


## 8 Progression, reasons, attributes

In [ ]:
cols = ["Land", "document", "page", "statement", "codes", "tools", "level", "grades_verbatim", "course", "niveau", "binding", "quote"]
PROGRESSION = S[S.code_progression][cols]
print(f"{len(PROGRESSION)} statements coded as transition/progression:")
display(PROGRESSION)

REASONS = S[S.reason.str.strip() != ""][["Land", "document", "page", "codes", "tools", "level", "binding", "reason"]]
print(f"\n{len(REASONS)} statements with a reason given in the document:")
display(REASONS)

In [ ]:
# 'suitable / didactically reduced / simple / age-appropriate' without further specification:
# how often is such a requirement made, and how often is it qualified by a reason?
vague = STATES[STATES.code_geeignet | STATES.code_einfach | STATES.code_altersangemessen]
print(f"{vague.Land.nunique()} states / {len(vague)} statements require 'suitable/simple/age-appropriate' tools; "
      f"{(vague.reason.str.strip() != '').sum()} of these statements give a reason.")
display(vague[["Land", "document", "level", "binding", "codes", "reason", "quote"]])

## 9 Recommendations (GI / KMK) vs. states

In [ ]:
display(REC[["Land", "document", "page", "statement", "codes", "tools", "level", "grades_verbatim", "niveau", "binding", "quote"]])
rec_types = overview(REC, TOOL_TYPE_CODES)
display(rec_types)

## 10 Export
* `output/curricula_synthesis.xlsx` – all tables of this notebook
* `output/tex/*.tex` – LaTeX tables (booktabs) for the thesis
* `output/fig/*.pdf|png` – figures

In [ ]:
with pd.ExcelWriter(OUT / "curricula_synthesis.xlsx") as xw:
    S.drop(columns=[c for c in S.columns if c.startswith("code_")]).assign(
        codes=lambda d: d.codes.map("; ".join), tools=lambda d: d.tools.map("; ".join),
        grades=lambda d: d.grades.map(lambda g: ", ".join(map(str, g)))).to_excel(xw, sheet_name="statements", index=False)
    CORPUS_DOCS.to_excel(xw, sheet_name="corpus", index=False)
    OVERVIEW_TYPES.to_excel(xw, sheet_name="overview_tool_types", index=False)
    TYPE_COUNTS.to_excel(xw, sheet_name="tool_type_counts", index=False)
    OVERVIEW_ATTR.to_excel(xw, sheet_name="overview_attributes", index=False)
    TOOL_SUMMARY.to_excel(xw, sheet_name="named_tools", index=False)
    TOOL_MATRIX.to_excel(xw, sheet_name="named_tools_by_state")
    TOOL_CATEGORIES.to_excel(xw, sheet_name="tool_categories")
    GRADES_ALL.to_excel(xw, sheet_name="grades_all")
    GRADES_BINDING.to_excel(xw, sheet_name="grades_binding")
    GRADES_EXPLICIT.to_excel(xw, sheet_name="grades_explicit")
    STATE_GRADES.to_excel(xw, sheet_name="state_x_grade")
    TOOLS_BY_GRADE.to_excel(xw, sheet_name="tools_by_grade")
    PROGRESSION.assign(codes=lambda d: d.codes.map("; ".join), tools=lambda d: d.tools.map("; ".join)).to_excel(xw, sheet_name="progression", index=False)
    REASONS.assign(codes=lambda d: d.codes.map("; ".join), tools=lambda d: d.tools.map("; ".join)).to_excel(xw, sheet_name="reasons", index=False)
    if len(PASSAGES_CODED[PASSAGES_CODED.relevant == "unklar"]):
        PASSAGES_CODED[PASSAGES_CODED.relevant == "unklar"].to_excel(xw, sheet_name="open_unklar", index=False)
print("written:", OUT / "curricula_synthesis.xlsx")

In [ ]:
def tex_escape(s):
    return str(s).replace("&", r"\&").replace("%", r"\%").replace("_", r"\_").replace("#", r"\#")

def to_tex(df, fname, caption, label, colspec=None, notes=None, size=r"\small"):
    df = df.copy()
    body = df.to_latex(index=False, escape=False, column_format=colspec or "l" * df.shape[1])
    body = body.replace("✓", r"\checkmark")
    note = ("\n\\begin{tablenotes}\\smaller\n" + "\n".join(f"\\item {n}" for n in notes) + "\n\\end{tablenotes}") if notes else ""
    tex = (f"\\begin{{table}}\n\\centering\n{size}\n\\begin{{threeparttable}}\n\\caption{{{caption}}}\\label{{{label}}}\n"
           f"{body}{note}\n\\end{{threeparttable}}\n\\end{{table}}\n")
    (OUT / "tex" / fname).write_text(tex, encoding="utf-8")
    print("written:", fname)

# Table: tool types by state and level (replaces tab:curricula in the thesis)
def sup(cell):
    """'x (elective/LK)' -> 'x\textsuperscript{e,LK}' to keep the table narrow."""
    m = re.match(r"^(.*?) \((.*)\)$", cell)
    if not m:
        return cell
    short = {"elective": "e", "LK": "LK", "adv.": "a"}
    return m.group(1) + r"\textsuperscript{" + ",".join(short.get(p, p) for p in m.group(2).split("/")) + "}"

t = OVERVIEW_TYPES.pivot(index="Land", columns="level", values=list(TOOL_TYPE_CODES.values())).fillna("")
t = t.reindex(columns=[(c, lv) for c in TOOL_TYPE_CODES.values() for lv in ["Sek I", "Sek II"]], fill_value="")
lines = [r"\begin{tabular}{l" + "cc" * len(TOOL_TYPE_CODES) + "}", r"\toprule",
         " & " + " & ".join(rf"\multicolumn{{2}}{{c}}{{{c}}}" for c in TOOL_TYPE_CODES.values()) + r" \\",
         "".join(rf"\cmidrule(lr){{{2 + 2 * i}-{3 + 2 * i}}}" for i in range(len(TOOL_TYPE_CODES))),
         "State & " + " & ".join(["Sek I & Sek II"] * len(TOOL_TYPE_CODES)) + r" \\", r"\midrule"]
# Joint Sek I curricula: one row per state, Sek I cells merged across the member states.
JOINT_SEK1 = {"Berlin&Brandenburg": ["Berlin", "Brandenburg"]}
fmt = lambda v: sup(v).replace("✓", r"\checkmark")
members = {m: j for j, ms in JOINT_SEK1.items() for m in ms}
for land in sorted((set(t.index) - set(JOINT_SEK1)) | set(members)):
    cells = []
    for (_, lv) in t.columns:
        joint = members.get(land)
        if joint and lv == "Sek I":
            first = JOINT_SEK1[joint][0] == land
            v = t.loc[joint, (_, lv)] if joint in t.index else ""
            cells.append(r"\multirow{%d}{*}{%s}" % (len(JOINT_SEK1[joint]), fmt(v)) if first else "")
        else:
            cells.append(fmt(t.loc[land, (_, lv)]) if land in t.index else "")
    lines.append(tex_escape(STATE_LABEL.get(land, land)) + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
notes = [r"\checkmark: mandatory, +: recommended, x: example, ?: bindingness not specified; strongest bindingness per state and level.",
         r"Only for elective courses (\textsuperscript{e}), advanced courses (\textsuperscript{LK}) or the advanced level of demand (\textsuperscript{a}).",
         r"Berlin and Brandenburg share the curriculum for Sekundarstufe~I (merged cells), but have separate curricula for Sekundarstufe~II."]
tex = "\n".join([r"\begin{table}", r"\centering", r"\small", r"\begin{threeparttable}",
                 r"\caption{Programming tool types required by German state curricula, by level}\label{tab:curricula-tool-types}",
                 *lines, r"\begin{tablenotes}\smaller", *[r"\item " + n for n in notes],
                 r"\end{tablenotes}", r"\end{threeparttable}", r"\end{table}", ""])
(OUT / "tex" / "tab_curricula_tool_types.tex").write_text(tex, encoding="utf-8")
print("written: tab_curricula_tool_types.tex")

# Table: named tools
nt = TOOL_SUMMARY[["tool", "category", "PL/PE", "states", "states_binding", "strongest", "levels"]].copy()
nt["strongest"] = nt["strongest"].map(BINDING_TEX)
nt.columns = ["Tool", "Category", "PL/PE", "States", "Mand.", "Max.", "Level"]
to_tex(nt.map(tex_escape), "tab_curricula_named_tools.tex",
       "Programming tools named in German state curricula", "tab:curricula-named-tools", "lllcccl",
       [r"States: number of states naming the tool; Mand.: of these, states in which it is mandatory; Max.: strongest bindingness (\checkmark: mandatory, +: recommended, x: example, ?: not specified)."])

# Table: requirements by grade (all statements)
gt = GRADES_ALL[GRADES_ALL.sum(axis=1) > 0].replace(0, "").reset_index().rename(columns={"index": "Requirement"})
gt.columns = [str(c) for c in gt.columns]
to_tex(gt.map(tex_escape), "tab_curricula_grades.tex", "Number of states with programming tool requirements, by grade",
       "tab:curricula-grades", "l" + "c" * (gt.shape[1] - 1),
       ["Grades partly derived from the level (Sek I: 5--10, Berlin/Brandenburg 7--10; Sek II: 11--13) where no explicit grade is given."])

In [ ]:

# Table: tool types by grade, explicit grade information only
eg = EXPLICIT_GRADES.copy()
eg.index = [tex_escape(i) for i in eg.index]
eg = eg.reset_index().rename(columns={"index": "State"})
eg.columns = [str(c) for c in eg.columns]
to_tex(eg, "tab_curricula_grades_explicit.tex",
       "Programming tool types by grade where curricula state grades explicitly", "tab:curricula-grades-explicit",
       "l" + "c" * (eg.shape[1] - 1),
       ["B: block-based, V: visual, T: text-based; upper case: mandatory, lower case: recommended, example or not specified.",
        "Only statements with explicit grade information; GI: recommendation of the Gesellschaft für Informatik."])


In [ ]:
# Appendix: all collected documents with corpus decision (source: 00_sources/corpus_include.csv)
reg = pd.read_csv(CURRICULA / "00_sources" / "corpus_include.csv")
reg["doc"] = reg.rel_path.str[:-5]
reg = reg[reg.doc.isin(DOCS)].copy()
reg["Land"] = reg.doc.map(state_of)
reg["included"] = reg.doc.map(in_corpus)
reg["statements"] = reg.doc.map(S.groupby("doc").size()).fillna(0).astype(int)
undecided = reg[~reg.included & reg.reason.fillna("").str.startswith("excluded")]
if len(undecided):
    print("WARNING: exclusions without specific reason:", list(undecided.doc))
assert reg.bibkey.notna().all(), f"documents without bibkey: {list(reg.doc[reg.bibkey.isna()])}"

def tex_text(s):
    return tex_escape(s).replace("Sek. ", r"Sek.\ ").replace("–", "--")

LAND_LABEL = STATE_LABEL

def rows_for(part, with_reason):
    part = part.assign(_rec=part.Land.isin(RECOMMENDATION_GROUPS)).sort_values(["_rec", "Land", "title"])
    out, prev = [], None
    for r in part.itertuples():
        land = LAND_LABEL.get(r.Land, r.Land) if r.Land != prev else ""
        prev = r.Land
        # each document is cited with its entry in diss.bib (column bibkey of the register)
        doc = (tex_text(r.title) + r" \parencite{" + r.bibkey + "}"
               + (r" \newline \emph{" + tex_text(r.reason) + "}" if with_reason else ""))
        count = str(r.statements) if not with_reason else ""
        out.append(f"{tex_text(land)} & {doc} & {count} " + r"\\")
    return out

# Only the documents of the corpus are listed; the excluded documents remain documented in the register.
inc, exc = reg[reg.included], reg[~reg.included]
head = [r"\toprule", r"State & Document & $n$ \\", r"\midrule"]
lines = ([r"\begin{longtable}{@{}p{0.28\linewidth}p{0.6\linewidth}r@{}}",
          r"\caption{Curriculum documents of the corpus}\label{tab:curricula-documents}\\"]
         + head + [r"\endfirsthead"] + head + [r"\endhead", r"\bottomrule", r"\endlastfoot"]
         + rows_for(inc, False)
         + [r"\end{longtable}", ""])
(OUT / "tex" / "tab_curricula_documents.tex").write_text("\n".join(lines), encoding="utf-8")
print(f"written: tab_curricula_documents.tex ({len(inc)} included; {len(exc)} excluded, not listed)")
